# MR3b — Hourly Final Join (Reduce-Side)
**Inputs**: `mr3/hourly_crash_mr.csv` + `mr3/address_store_mr.csv` + `mr3/type_count_store_mr.csv`
**Output**: `mr3/mr_hourly_final.csv`

In [1]:
import json
import pandas as pd
from collections import defaultdict

CRASH_INPUT  = 'hourly_crash_mr.csv'
ADDR_INPUT   = 'address_store_mr.csv'
TYPE_INPUT   = 'type_count_store_mr.csv'
OUTPUT       = 'mr_hourly_final.csv'

crashes = pd.read_csv(CRASH_INPUT)
addr    = pd.read_csv(ADDR_INPUT)
types   = pd.read_csv(TYPE_INPUT)

print(f'Hourly crash rows: {len(crashes):,}  |  zones: {crashes["zone_id"].nunique():,}')
print(f'Store zones      : {len(addr):,}')
print(f'Type rows        : {len(types):,}')

Hourly crash rows: 2,708  |  zones: 1,490
Store zones      : 1,823
Type rows        : 6,576


In [ ]:
# ── MAP ───────────────────────────────────────────────────────────────────────
type_by_zone = defaultdict(dict)
for _, row in types.iterrows():
    type_by_zone[str(row['zone_id'])][row['method_of_operation']] = int(row['count'])

def mapper(row, source):
    zid = str(row['zone_id'])
    if source == 'crash':
        return zid, {
            'src':         'crash',
            'hour':        int(row['hour']),
            'avg_crashes': row['avg_crashes'],
            'avg_injured': row['avg_injured'],
            'avg_killed':  row['avg_killed'],
        }
    else:
        return zid, {
            'src':               'store',
            'store_count':       row['store_count'],
            'active_licenses':   row['active_licenses'],
            'outdated_licenses': row['outdated_licenses'],
            'type_counts_json':  json.dumps(type_by_zone.get(zid, {})),
        }

mapped  = [mapper(row, 'crash') for _, row in crashes.iterrows()]
mapped += [mapper(row, 'store') for _, row in addr.iterrows()]

print(f'Total mapped pairs: {len(mapped):,}')

Total mapped pairs: 4,531


In [3]:
# ── SHUFFLE ───────────────────────────────────────────────────────────────────
# Group all tagged records by zone_id

shuffled = defaultdict(list)
for zone_id, record in mapped:
    shuffled[zone_id].append(record)

print(f'Unique zone_ids after shuffle: {len(shuffled):,}')

Unique zone_ids after shuffle: 2,289


In [ ]:
# ── REDUCE ────────────────────────────────────────────────────────────────────────────────
_EMPTY_STORE = {
    'store_count':       0,
    'active_licenses':   0,
    'outdated_licenses': 0,
    'type_counts_json':  '{}',
}

def reducer(zone_id, records):
    store_rec  = next((r for r in records if r['src'] == 'store'), None)
    crash_recs = [r for r in records if r['src'] == 'crash']
    if not crash_recs:
        return []
    sr = store_rec if store_rec else _EMPTY_STORE
    return [
        {
            'zone_id':           zone_id,
            'hour':              c['hour'],
            'avg_crashes':       c['avg_crashes'],
            'avg_injured':       c['avg_injured'],
            'avg_killed':        c['avg_killed'],
            'store_count':       sr['store_count'],
            'active_licenses':   sr['active_licenses'],
            'outdated_licenses': sr['outdated_licenses'],
            'type_counts_json':  sr['type_counts_json'],
        }
        for c in crash_recs
    ]

rows = []
for zone_id, records in shuffled.items():
    rows.extend(reducer(zone_id, records))

out = pd.DataFrame(rows).sort_values(['zone_id','hour']).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

crash_only_zones = out[out['store_count'] == 0]['zone_id'].nunique()
print(f'Output rows      : {len(out):,}')
print(f'Unique zones     : {out["zone_id"].nunique():,}')
print(f'Hours covered    : {sorted(out["hour"].unique())}')
print(f'Crash-only zones : {crash_only_zones:,}  (store_count=0, retained via left join)')
print(f'Saved → {OUTPUT}')
out.head().transpose()

Output rows      : 2,708
Unique zones     : 1,490
Hours covered    : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
Crash-only zones : 466  (store_count=0, retained via left join)
Saved → mr_hourly_final.csv


,0,1,2,3,4
zone_id,8a2a1008800ffff,8a2a1008800ffff,8a2a1008802ffff,8a2a1008802ffff,8a2a10088047fff
hour,1,20,0,23,3
avg_crashes,1.0,1.0,1.0,1.0,1.0
avg_injured,2.0,1.0,0.0,1.0,0.0
avg_killed,0.0,0.0,0.0,0.0,0.0
store_count,0,0,0,0,0
active_licenses,0,0,0,0,0
outdated_licenses,0,0,0,0,0
type_counts_json,{},{},{},{},{}
